In [1]:
!wget https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
!unzip -q maestro-v3.0.0-midi.zip

--2026-10-14 21:14:34--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.118.207, 172.217.70.207, 142.250.4.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.118.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58416533 (56M) [application/octet-stream]
Saving to: ‘maestro-v3.0.0-midi.zip.1’

maestro-v3.0.0-midi 100%[===================>]  55.71M  16.5MB/s    in 4.1s    

2026-10-14 21:14:40 (13.7 MB/s) - ‘maestro-v3.0.0-midi.zip.1’ saved [58416533/58416533]

replace maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [2]:
!apt-get update -qq
!apt-get install -y -qq fluidsynth fluid-soundfont-gm
!pip install -q pyfluidsynth pretty_midi

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
import torch
import pretty_midi
from pathlib import Path
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

### READING FILES AND TOKENIZE

In [4]:
files = sorted(Path("maestro-v3.0.0").rglob("*.midi"))
print(len(files))   # should be around 1,276
print(files[0])

1276
maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi


In [5]:
"""
CALCULATE MIN AND MAX IF NECESSARY
MAX = 0
MIN = float('inf')

for file in tqdm(files):
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = pm.instruments[0]
    for note in piano.notes:
        if MAX < note.pitch: MAX = note.pitch
        if MIN > note.pitch: MIN = note.pitch"""

"\nCALCULATE MIN AND MAX IF NECESSARY\nMAX = 0\nMIN = float('inf')\n\nfor file in tqdm(files):\n    pm = pretty_midi.PrettyMIDI(str(file))\n    piano = pm.instruments[0]\n    for note in piano.notes:\n        if MAX < note.pitch: MAX = note.pitch\n        if MIN > note.pitch: MIN = note.pitch"

In [6]:
MAX = 108
MIN = 21

In [7]:
import numpy as np
import torch

SCALE = 0.001
MAXTIME = 100
pieces = []

class Note():
    def __init__(self, pitch, time, timing):
        self.pitch = pitch
        self.time = int(round(time / SCALE))
        self.timing = timing

def make_tokens(file):
    j = k = 0
    notes = []
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)
    for note in piano.notes:
            note_1 = Note(note.pitch, note.start, 'START')
            note_2 = Note(note.pitch, note.end, 'END')
            notes.append(note_1)
            notes.append(note_2)
    sorted_notes = sorted(notes, key=lambda x: x.time)

    final_notes = []
    for i in range(len(sorted_notes) - 1):
        final_notes.append((sorted_notes[i].timing, sorted_notes[i].pitch))
        time_step = sorted_notes[i+1].time - sorted_notes[i].time
        if time_step != 0:
            while time_step > MAXTIME:
                final_notes.append(('TIME', MAXTIME))
                time_step -= MAXTIME
            final_notes.append(('TIME', time_step))

    final_notes.append((sorted_notes[-1].timing, sorted_notes[-1].pitch))
    return final_notes

In [8]:
token_sqs = [make_tokens(file) for file in tqdm(files)]

100%|██████████| 1276/1276 [05:56<00:00,  3.57it/s]


In [9]:
token_sqs_lens = [len(token_sq) for token_sq in token_sqs]

MAX_PIECE = max(token_sqs_lens)
MIN_PIECE = min(token_sqs_lens)
MEAN_PIECE = sum(token_sqs_lens)/len(token_sqs_lens)

print(f"MAX: {MAX_PIECE} MIN: {MIN_PIECE} MEAN: {MEAN_PIECE}")

MAX: 109069 MIN: 853 MEAN: 24211.17789968652


In [10]:
def make_vocab():
    vocab = {'PAD': 0, 'BOS': 1, 'EOS': 2}
    for kind in ('START', 'END'):
        for p in range(MIN, MAX+1):
            vocab[(kind, p)] = len(vocab)
    for t in range(1, MAXTIME+1):
        vocab[('TIME', t)] = len(vocab)
    return vocab

def make_ids(tokens, vocab):
    ids = [vocab['BOS']]
    for key, val in tokens:
        ids.append(vocab[(key, val)])
    ids.append(vocab['EOS'])
    return ids

In [11]:
#CREATE VOCAB TOKEN -> IDS
vocab = make_vocab()
vocab_size = len(vocab)

### CREATING THE DATA TENSOR

In [12]:
#CREATING THE TENSOR
sq_len = 100

sqs = []
for sq in tqdm(token_sqs):
    sq = torch.tensor(make_ids(sq, vocab), dtype=torch.long)[:sq_len]
    sq = torch.nn.functional.pad(sq, (0, sq_len - len(sq)), value=0)
    sqs.append(sq)

data = torch.stack(sqs)
data = data.to(device)

100%|██████████| 1276/1276 [00:05<00:00, 216.89it/s]


In [13]:
n = 0.9

data_train = data[:int(len(data)*n)]
data_test = data[int(len(data)*n):]

In [14]:
from torch.utils.data import DataLoader
def make_xy(samples):
    samples = torch.stack(samples)
    return samples[..., :-1], samples[..., 1:]

data_train_loader = DataLoader(data_train, batch_size=16, shuffle=True, collate_fn=make_xy)
data_test_loader = DataLoader(data_test, batch_size=16, shuffle=True, collate_fn=make_xy)

### BIGRAM MODEL

In [43]:
import torch.nn as nn
torch.manual_seed(1502)

class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        logits = self.token_embedding_table(idx)
        B, N, L = logits.shape #Inside each batch, each note is now a list of logits
        logits = logits.reshape(B*N, L)
        return logits

In [44]:
m = BigramModel(vocab_size)
m = m.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=m.parameters(), lr=1e-3)

for epoch in tqdm(range(100)):
    for xb, yb in data_train_loader:

        B, N = yb.shape
        yb = yb.reshape(B*N)

        y_pred = m(xb)
        loss = loss_fn(y_pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
print(loss.item())

100%|██████████| 100/100 [00:06<00:00, 14.96it/s]

3.589205503463745


### AUDIO GENERATION

In [15]:
from IPython.display import Audio, display

def generate_bigrammodel(bigrammodel, max_tokens, idx=1): #idx=1=BOS
    bigrammodel = bigrammodel.to('cpu')
    sq = [idx]
    for _ in range(max_tokens):
        idx = torch.tensor(idx, dtype=torch.long).unsqueeze(dim=-1).unsqueeze(dim=-1)
        logits = bigrammodel(idx)
        logits = logits
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq.append(idx)
    return sq

def generate_selfattention(model, max_tokens, idx=1):
    sq = [idx]
    for _ in range(max_tokens):
        idx = torch.tensor(sq, dtype=torch.long).unsqueeze(dim=0).to(device)
        logits = model(idx)
        logits = logits[-1]
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq.append(idx)
    return sq

def ids_to_tokens(ids, vocab):
    inv = {v: k for k, v in vocab.items()}
    tokens = []
    for i in ids:
        tok = inv[int(i)]
        if tok in ('PAD', 'BOS'):
            continue
        if tok == 'EOS':
            break
        tokens.append(tok)
    return tokens

def tokens_to_midi(tokens, scale, path='out.mid', velocity=80):
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)
    t = 0                                # clock, in quantized ticks
    active = {}                          # pitch -> start tick

    def close(pitch, end_tick):
        start = active.pop(pitch)
        if end_tick > start:             # skip zero-length notes
            inst.notes.append(pretty_midi.Note(
                velocity=velocity, pitch=pitch,
                start=start * scale, end=end_tick * scale))

    for kind, val in tokens:
        if kind == 'TIME':
            t += val
        elif kind == 'START':
            if val in active:            # same pitch already sounding
                close(val, t)
            active[val] = t
        elif kind == 'END':
            if val in active:            # ignore ENDs with no matching START
                close(val, t)

    for pitch in list(active):           # close notes left hanging at the end
        close(pitch, t)

    pm.instruments.append(inst)
    pm.write(path)
    return pm

def make_audio(pm):
    sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
    display(Audio(audio, rate=44100))

def ids_to_audio(ids, vocab=vocab, scale=SCALE, path='out.mid', velocity=80):
    tokens = ids_to_tokens(ids, vocab)
    pm = tokens_to_midi(tokens, scale, path, velocity)
    make_audio(pm)

In [55]:
ids = generate(bigrammodel=m, max_tokens=1000)
tokens = ids_to_tokens(ids, vocab)
pm = tokens_to_midi(tokens, scale=SCALE)
make_audio(pm)

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

In [22]:
pm = pretty_midi.PrettyMIDI(str(files[0]))
sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
display(Audio(audio, rate=44100))

/tmp/ipykernel_3803/1570866834.py:3: DeprecationWarning: The parameter 'sf2_path' is deprecated, please use 'synthesizer' instead.
  audio = pm.fluidsynth(fs=44100, sf2_path=sf2)


KeyboardInterrupt: 

### SLEF ATTENTION

In [16]:
import torch.nn as nn
torch.manual_seed(1502)
n_embed = 512
head_size=512

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(sq_len, sq_len)))

    def forward(self, x):
        B,T,N = x.shape #B, T, N_EMBED
        k = self.key(x)   # (B,T,H)
        q = self.query(x) # (B,T,H)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * N**-0.5 # (B, T, H) @ (B, H, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = nn.functional.softmax(wei, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,H)
        out = wei @ v # (B, T, T) @ (B, T, H) -> (B, T, H)
        return out

class BigramModel2(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(sq_len, n_embed)
        self.head = Head(head_size)
        self.linear = nn.Linear(n_embed, vocab_size)

    def forward(self, idx):
        B, T = idx.shape

        token_embed = self.token_embedding_table(idx) #B, T, N_EMBED
        pos_embed = self.position_embedding_table(torch.arange(T, device=device)) #T, C is the same for every batch
        x = token_embed + pos_embed
        x = self.head(x)
        logits = self.linear(x)

        B, T, L = logits.shape #BATCH, TOKEN, LOGIT
        logits = logits.reshape(B*T, L)
        return logits

In [17]:
def training(model, loss_fn, optimizer, epochs, dataloader):
    for epoch in tqdm(range(epochs)):
        for xb, yb in dataloader:
            B, N = yb.shape
            yb = yb.reshape(B*N)

            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return loss.item()

In [18]:
m = BigramModel2(vocab_size).to(device)

In [23]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=m.parameters(), lr=1e-3)
epochs = 500
print(training(
    model=m,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=epochs,
    dataloader=data_train_loader
))

  0%|          | 0/500 [00:00<?, ?it/s]


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [22]:
ids = generate_selfattention(model=m, max_tokens=100)
tokens = ids_to_tokens(ids, vocab)
pm = tokens_to_midi(tokens, scale=SCALE)
make_audio(pm)

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [111]:
sq = [1, 2]
idx = torch.tensor(sq, dtype=torch.long).unsqueeze(dim=0).to('cpu')
idx

tensor([[1, 2]])

In [88]:
probs

tensor([[8.7119e-10, 2.6292e-10, 4.2899e-10, 1.4731e-06, 1.1999e-04, 1.3539e-08,
         4.9066e-04, 1.0638e-03, 2.5423e-03, 2.2883e-02, 1.5287e-03, 1.9681e-04,
         4.9731e-03, 3.5419e-03, 2.3869e-03, 1.0576e-03, 3.1034e-03, 2.2877e-04,
         8.2952e-02, 2.4864e-02, 1.4547e-03, 1.5191e-02, 2.2810e-02, 7.5309e-03,
         2.7608e-02, 1.3222e-02, 1.7437e-03, 9.2135e-03, 2.7051e-02, 7.5567e-03,
         5.0582e-02, 2.5692e-02, 2.3231e-02, 1.0979e-02, 9.2356e-03, 1.1393e-02,
         2.2273e-02, 9.9725e-03, 1.6730e-03, 1.6105e-02, 4.3414e-03, 6.1454e-03,
         2.1096e-02, 1.8890e-03, 6.7169e-03, 3.6837e-02, 8.5363e-03, 2.0759e-02,
         2.2020e-02, 6.1056e-02, 7.7925e-03, 1.6897e-02, 9.6518e-03, 2.1201e-02,
         2.0778e-02, 6.7957e-03, 8.4472e-03, 3.5314e-02, 2.4606e-03, 1.8622e-02,
         9.8978e-03, 9.8213e-02, 2.1307e-03, 8.1236e-03, 1.6144e-02, 5.8276e-04,
         4.7900e-03, 2.2537e-04, 3.8312e-03, 1.5113e-03, 2.3363e-03, 6.1648e-04,
         3.4792e-03, 8.9027e

In [ ]:
make_audio(pm)